# 두 선수의 캐치에서 양쪽 무릎이 각각 반대쪽 예측에 가까운가?

기본 검증 영상의 **c04 78번**과 **c08 58번**을 비교합니다. 두 장 모두 1차로 기록한 캐치이며, 카메라에 가까운 다리는 선수 자신의 **왼쪽**입니다. 같은 C그룹 두 선수의 한 장씩을 보는 탐색 실험입니다.

### 1. 예측 없는 원본 확인

c04 왼쪽 무릎은 이전에 직접 찍은 1차 좌표를 그대로 불러옵니다. 원본과 판독 탭에서는 그 점도 숨깁니다. c08의 캐치는 2차 판독에서 59번으로 기록됐으며, 여기서는 두 선수 모두 1차 기준을 씁니다.

In [ ]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from ultralytics import YOLO

from wlcv.annotation.pose_points import pose_point_editor
from wlcv.config import ROOT, VIDEOS
from wlcv.data.frames import read_rgb
from wlcv.evaluation.bilateral_knees import SIDES, draw_assignments, evaluate
from wlcv.evaluation.catch_knee_sides import (
    CROP, FOLDER, ORIGINAL, SCENES, check_scenes, opposite_pattern, read_catches, repeat_summary,
)

check_scenes()
manual = read_catches(FOLDER)
weights = ROOT / "models/yolo26n-pose.pt"
output = ROOT / "outputs/catch_knee_sides"
fig, axes = plt.subplots(1, 2, figsize=(6, 5), layout="constrained")
for ax, (video, frame) in zip(axes, SCENES.items()):
    ax.imshow(read_rgb(VIDEOS / f"{video}.mp4", frame))
    ax.set_title(f"{video} #{frame}")
    ax.axis("off")
plt.show()

### 2. 세 개의 무릎 판독

탭에 적힌 선수와 방향을 확인하고 **무릎 관절선**을 한 번 클릭하세요. 확대 화면의 클릭은 원본 픽셀로 자동 저장됩니다. 겹쳐서 직접 짚기 어려우면 **가려서 판독 불가** 또는 **위치 불확실**을 누르세요. 위치를 추측해서 찍을 필요가 없습니다. 각 탭에서는 그 탭에 직접 저장한 점만 다시 보입니다.

In [ ]:
panels, titles = [], []
for video, frame in SCENES.items():
    for side, label in SIDES.items():
        if (video, side) in ORIGINAL:
            continue
        panel = pose_point_editor(VIDEOS / f"{video}.mp4", frame, video.split("_")[0], side,
                                  split="validation", annotations=FOLDER / side, joints=("무릎",),
                                  frame_label=f"{video[:3]} {frame}번", direct_only=True, crop=CROP)
        panels.append(panel)
        titles.append(f"{video[:3]} {frame}번 · {label}")
tabs = widgets.Tab(children=panels)
for index, title in enumerate(titles):
    tabs.set_title(index, title)
display(tabs)

### 3. 세 점을 저장한 뒤 결과 확인

이 셀을 다시 실행하면 **같은 수동점에서** YOLO의 같은 쪽과 반대쪽 예측까지 거리를 비교합니다. 선수의 왼쪽 점에서도, 오른쪽 점에서도 각각 반대쪽 거리가 더 짧으면 그 선수에게서 패턴을 **관찰**한 것입니다. 두 선수 모두 이 조건을 만족해야 이 두 캐치에서 **반복 관찰**로 셉니다.

표에는 각 무릎의 거리 차이(같은 쪽 − 반대쪽, 원본 px), 네 개별 거리, 두 연결 거리 합을 표시합니다. 거리 동률은 ‘더 가까움’을 충족하지 않습니다. 판독 불가, 수행자 미탐·복수, 좌우 무릎 예측 중 하나라도 신뢰도 0.50 미만이면 해당 선수의 판단을 보류합니다. 기존 a05의 ‘판정’ 열은 두 교차 거리가 같은 쪽 거리 두 개보다 모두 짧아야 하는 더 엄격한 검사를 별도로 보여줍니다.

원본 YOLO26n-pose로 전체 프레임을 추론합니다(입력 640, CPU, 사람 신뢰도 0.25). 그림의 원은 수동점, ×는 모델 예측이며 청록은 왼쪽, 노랑은 오른쪽입니다.

In [ ]:
manual = read_catches(FOLDER)
saved = manual["판독"].ne("미검수").sum() - len(ORIGINAL)
print(f"신규 판독 {saved}/3개 저장 · 기존 c04 왼쪽 1개")
if saved < 3:
    display(manual[["영상", "프레임", "방향", "판독"]])
    print("세 탭을 모두 저장한 뒤 이 셀을 다시 실행하세요.")
else:
    if not weights.is_file():
        raise FileNotFoundError(f"원본 자세 모델을 확인하세요. {weights}")
    model = YOLO(weights)
    output.mkdir(parents=True, exist_ok=True)
    all_coords, all_scores = [], []
    for video, frame in SCENES.items():
        source = VIDEOS / f"{video}.mp4"
        labels = manual.loc[manual["영상"] == video].drop(columns="영상")
        coords, scores = evaluate(model, source, labels, frames=[frame])
        scores = scores.join(scores.apply(opposite_pattern, axis=1, result_type="expand"))
        figure = draw_assignments(source, coords, scores, frames=[frame], roi=CROP)
        figure.suptitle(f"{video}  |  catch")
        figure.savefig(output / f"{video}_assignments.png", dpi=150, bbox_inches="tight")
        plt.show()
        all_coords.append(coords.assign(영상=video))
        all_scores.append(scores.assign(영상=video))
    report = pd.concat(all_scores, ignore_index=True)
    print(repeat_summary(report))
    display(report[["영상", "프레임", "왼쪽 거리 차(px)", "오른쪽 거리 차(px)", "두 무릎 모두 반대쪽"]].round(2))
    display(report.round(2))
    coordinates = pd.concat(all_coords, ignore_index=True)
    display(coordinates.round(2))
    report.to_csv(output / "catch_assignments.csv", index=False)
    coordinates.to_csv(output / "catch_coordinates.csv", index=False)
    print(f"결과 저장 위치: {output}")